# ЛР2: Эксперимент 2.3 — Влияние размера истории L-BFGS

Требуемые графики:
- относительный квадрат нормы градиента vs итерации (log-scale);
- относительный квадрат нормы градиента vs время (log-scale);
- итоговое время vs размер истории.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from oracles import LogCoshL2Oracle, ExponentialLossL2Oracle
from optimization import lbfgs

rng = np.random.default_rng(42)

In [ ]:
# При наличии своего LIBSVM-датасета подставьте загрузку здесь.
# Ниже fallback на синтетику, чтобы ноутбук был самодостаточным.
m, n = 4000, 300
A = rng.normal(size=(m, n))
true_w = rng.normal(size=n)
b_reg = A @ true_w + 0.1 * rng.normal(size=m)
b_clf = np.sign(A @ true_w + 0.3 * rng.normal(size=m))
b_clf[b_clf == 0] = 1
A_sp = sparse.csr_matrix(A)

def matvec_Ax(x):
    return A_sp.dot(x)

def matvec_ATx(y):
    return A_sp.T.dot(y)

def matmat_ATsA(s):
    return A_sp.T.dot(sparse.diags(s)).dot(A_sp).toarray()

In [ ]:
regcoef = 1.0 / m
oracle_reg = LogCoshL2Oracle(matvec_Ax, matvec_ATx, matmat_ATsA, b_reg, regcoef)
oracle_clf = ExponentialLossL2Oracle(matvec_Ax, matvec_ATx, matmat_ATsA, b_clf, regcoef)

memory_sizes = [0, 1, 5, 10, 50, 100]
x0 = np.zeros(n)

def run_suite(oracle, title):
    traces = {}
    for l in memory_sizes:
        _, msg, hist = lbfgs(
            oracle, x0,
            tolerance=1e-8,
            max_iter=300,
            memory_size=l,
            line_search_options={'method': 'Wolfe', 'c1': 1e-4, 'c2': 0.9, 'alpha_0': 1.0},
            trace=True,
        )
        traces[l] = (msg, hist)
    return traces

tr_reg = run_suite(oracle_reg, 'Regression')
tr_clf = run_suite(oracle_clf, 'Classification')

In [ ]:
def plot_curves(traces, name):
    plt.figure(figsize=(14, 4))
    
    plt.subplot(1, 3, 1)
    for l, (_, hist) in traces.items():
        g = np.array(hist['grad_norm'])
        rel = (g ** 2) / max(g[0] ** 2, 1e-32)
        plt.plot(rel, label=f'L={l}')
    plt.yscale('log')
    plt.xlabel('iteration')
    plt.ylabel(r'$||g_k||^2 / ||g_0||^2$')
    plt.title(f'{name}: vs iteration')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.subplot(1, 3, 2)
    total_times = []
    for l, (_, hist) in traces.items():
        g = np.array(hist['grad_norm'])
        rel = (g ** 2) / max(g[0] ** 2, 1e-32)
        t = np.array(hist['time'])
        plt.plot(t, rel, label=f'L={l}')
        total_times.append(t[-1])
    plt.yscale('log')
    plt.xlabel('time, sec')
    plt.ylabel(r'$||g_k||^2 / ||g_0||^2$')
    plt.title(f'{name}: vs time')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(memory_sizes, total_times, marker='o')
    plt.xlabel('history size L')
    plt.ylabel('total time, sec')
    plt.title(f'{name}: time vs L')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()

plot_curves(tr_reg, 'Regression')
plot_curves(tr_clf, 'Classification')

In [ ]:
# Оценка памяти и сложности одной итерации L-BFGS (без стоимости оракула)
# Память: O(nL), две истории s_i и y_i => примерно 2*n*L чисел.
# Two-loop рекурсия: O(nL) скалярных/векторных операций.
table = []
for L in memory_sizes:
    memory_numbers = 2 * n * L
    flops_unit = 8 * n * max(L, 1)
    table.append((L, memory_numbers, flops_unit))
table

## Что написать в отчёте
- Есть ли плато по итерациям при росте `L`.
- Оптимальное `L` по времени и почему слишком большой `L` может замедлять метод.
- Как могла бы измениться оптимальная `L` при более плохой обусловленности (сильная корреляция признаков).